In [1]:
!pip install yfinance

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 66.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 55.4 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 12.0
    Uninstalling websockets-12.0:
      Successfully uninstalled websockets-12.0
  Attempting uninstall: cffi
    Found existing installation: cffi 1.16.0
    Uninstalling cffi-1.16.0:
      Successfully uninstalled cffi-1.16.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.


In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DoubleType, LongType, BooleanType, TimestampType
)

BRONZE_TABLE = "bronze_price_ohlcv"

schema = StructType([
    StructField("symbol",              StringType(),    False),
    StructField("yahoo_symbol",        StringType(),    False),
    StructField("trade_date",          DateType(),      False),
    StructField("open",                DoubleType(),    True),
    StructField("high",                DoubleType(),    True),
    StructField("low",                 DoubleType(),    True),
    StructField("close",               DoubleType(),    True),
    StructField("volume",              LongType(),      True),
    StructField("source",              StringType(),    True),
    StructField("price_adjusted_flag", BooleanType(),   True),
    StructField("retrieved_at",        TimestampType(), True),
    StructField("run_id",              StringType(),    True),
])

# Create an empty Delta table with this schema
empty_df = spark.createDataFrame([], schema)

empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

# Confirm
display(spark.read.table(BRONZE_TABLE).printSchema())

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 4, Finished, Available, Finished, False)

root
 |-- symbol: string (nullable = true)
 |-- yahoo_symbol: string (nullable = true)
 |-- trade_date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- source: string (nullable = true)
 |-- price_adjusted_flag: boolean (nullable = true)
 |-- retrieved_at: timestamp (nullable = true)
 |-- run_id: string (nullable = true)



In [3]:
# ============================================================
# NB02_Price_Bronze_Ingestion
# Purpose:
#   Download historical and incremental adjusted OHLCV
#   for eligible stocks and Nifty 50 index.
#   Save to bronze_price_ohlcv.
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import uuid

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

# Read parameters from config file
PARAMS = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load("Files/config/model_parameters.csv")
    .toPandas()
    .set_index("parameter")["value"]
)

HISTORY_START_DATE = PARAMS.get("history_start_date", "2019-01-01")
MIN_PRICE_HISTORY_DAYS = int(PARAMS.get("min_price_history_days", 260))
RE_DOWNLOAD_WINDOW_DAYS = 10  # Re-download last 10 calendar days for safety

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow()  # keep as datetime, not a string


StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 5, Finished, Available, Finished, False)

In [4]:
df = spark.read.format("csv").option("header","true").load("Files/config/model_parameters.csv")
# df now is a Spark DataFrame containing CSV data from "Files/config/model_parameters.csv".
display(df)

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8df00102-8841-4dda-9b84-2e7117474df1)

In [5]:
# ------------------------------------------------------------
# 2. Load eligible universe
# ------------------------------------------------------------

universe = spark.sql("""
    SELECT
        symbol,
        yahoo_symbol,
        company_name,
        instrument_type
    FROM bronze_universe_snapshot
    WHERE instrument_type = 'EQUITY'
      AND eligible_for_stock_screen = true
""").toPandas()

market_index = spark.sql("""
    SELECT
        symbol,
        yahoo_symbol,
        company_name
    FROM bronze_universe_snapshot
    WHERE instrument_type = 'INDEX'
      AND symbol = 'NIFTY50'
""").toPandas()

# Combine stocks and index
all_symbols = pd.concat([universe, market_index], ignore_index=True)

print(f"Total symbols to fetch: {len(all_symbols)}")
print(f"Stocks: {len(universe)}")
print(f"Index: {len(market_index)}")



StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 7, Finished, Available, Finished, False)

Total symbols to fetch: 399
Stocks: 398
Index: 1


In [6]:
# ------------------------------------------------------------
# 3. Determine date range
# ------------------------------------------------------------
import pandas as pd
from datetime import date, timedelta

today = date.today()
history_start = pd.to_datetime(HISTORY_START_DATE, dayfirst=True).date()


# For incremental run: find the latest date already in bronze
try:
    latest_date = (
        spark.sql("""
            SELECT MAX(trade_date) AS max_date
            FROM bronze_price_ohlcv
        """)
        .toPandas()
        .iloc[0]["max_date"]
    )

    if pd.isna(latest_date):
        latest_date = history_start
    else:
        latest_date = latest_date.date()

except:
    latest_date = history_start

# Incremental start = latest date + 1 day
incremental_start = latest_date + timedelta(days=1)

# Re-download window: last RE_DOWNLOAD_WINDOW_DAYS days
re_download_start = max(
    history_start,
    today - timedelta(days=RE_DOWNLOAD_WINDOW_DAYS)
)

# Final start date = min(incremental_start, re_download_start)
final_start = min(incremental_start, re_download_start)

print(f"History start: {history_start}")
print(f"Latest date in bronze: {latest_date}")
print(f"Incremental start: {incremental_start}")
print(f"Re-download start: {re_download_start}")
print(f"Final start date: {final_start}")



StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 8, Finished, Available, Finished, False)

History start: 2019-01-01
Latest date in bronze: 2019-01-01
Incremental start: 2019-01-02
Re-download start: 2026-09-10
Final start date: 2019-01-02


In [7]:
# ------------------------------------------------------------
# 4. Download function
# ------------------------------------------------------------

def download_yahoo_data(yahoo_symbol, start_date, end_date):
    """
    Download adjusted OHLCV data from Yahoo Finance.
    Returns a DataFrame or None if failed.
    """
    try:
        data = yf.download(
            yahoo_symbol,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False
        )

        if len(data) == 0:
            print(f"  No data for {yahoo_symbol}")
            return None

        data = data.reset_index()
        data["yahoo_symbol"] = yahoo_symbol
        data = data[["Date", "yahoo_symbol", "Open", "High", "Low", "Close", "Volume"]]
        data.columns = ["trade_date", "yahoo_symbol", "open", "high", "low", "close", "volume"]

        return data

    except Exception as error:
        print(f"  Failed {yahoo_symbol}: {str(error)}")
        return None



StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 9, Finished, Available, Finished, False)

In [8]:
# ------------------------------------------------------------
# 5. Download data
# ------------------------------------------------------------

all_data = []
failed_symbols = []

for idx, row in all_symbols.iterrows():
    symbol = row["symbol"]
    yahoo_symbol = row["yahoo_symbol"]

    print(f"Fetching {symbol} ({yahoo_symbol})...")

    data = download_yahoo_data(
        yahoo_symbol,
        start_date=final_start,
        end_date=today
    )

    if data is not None:
        data["symbol"] = symbol
        data["source"] = "yfinance"
        data["price_adjusted_flag"] = True
        data["retrieved_at"] = RUN_TIMESTAMP
        data["run_id"] = RUN_ID
        all_data.append(data)

    else:
        failed_symbols.append(symbol)

    # Be polite to Yahoo
    time.sleep(0.5)



StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 10, Finished, Available, Finished, False)

Fetching POWERGRID (POWERGRID.NS)...
Fetching PREMIERENE (PREMIERENE.NS)...
Fetching PRESTIGE (PRESTIGE.NS)...
Fetching PFOCUS (PFOCUS.NS)...
Fetching RRKABEL (RRKABEL.NS)...
Fetching RHIM (RHIM.NS)...
Fetching RITES (RITES.NS)...
Fetching RADICO (RADICO.NS)...
Fetching RVNL (RVNL.NS)...
Fetching RAILTEL (RAILTEL.NS)...
Fetching RAINBOW (RAINBOW.NS)...
Fetching RKFORGE (RKFORGE.NS)...
Fetching REDINGTON (REDINGTON.NS)...
Fetching RELIANCE (RELIANCE.NS)...
Fetching RPOWER (RPOWER.NS)...
Fetching SJVN (SJVN.NS)...
Fetching SRF (SRF.NS)...
Fetching SAGILITY (SAGILITY.NS)...
Fetching SAILIFE (SAILIFE.NS)...
Fetching MOTHERSON (MOTHERSON.NS)...
Fetching SAPPHIRE (SAPPHIRE.NS)...
Fetching SARDAEN (SARDAEN.NS)...
Fetching SAREGAMA (SAREGAMA.NS)...
Fetching SCHAEFFLER (SCHAEFFLER.NS)...
Fetching SCHNEIDER (SCHNEIDER.NS)...
Fetching SCI (SCI.NS)...
Fetching SHREECEM (SHREECEM.NS)...
Fetching SHYAMMETL (SHYAMMETL.NS)...
Fetching ENRIN (ENRIN.NS)...
Fetching SIEMENS (SIEMENS.NS)...
Fetching SIGNA

In [9]:
BRONZE_SCHEMA = spark.table("bronze_price_ohlcv").schema

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 11, Finished, Available, Finished, False)

In [10]:
# ------------------------------------------------------------
# 6. Combine and validate
# ------------------------------------------------------------

if all_data:
    combined = pd.concat(all_data, ignore_index=True)

    # Force trade_date to a plain date (not datetime64) to match the
    # DATE column in bronze_price_ohlcv
    combined["trade_date"] = pd.to_datetime(combined["trade_date"]).dt.date
    combined["volume"] = combined["volume"].fillna(0).astype("int64")   # add this line
    combined = combined.drop_duplicates(
        subset=["symbol", "trade_date"],
        keep="last"
    )
    ...

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 12, Finished, Available, Finished, False)

In [11]:
if len(combined) > 0:
    # normalize types to exactly match the table before handing to Spark
    combined["trade_date"] = pd.to_datetime(combined["trade_date"]).dt.date
    combined["volume"] = combined["volume"].fillna(0).astype("int64")
    combined["retrieved_at"] = pd.to_datetime(combined["retrieved_at"])
    combined["open"] = combined["open"].astype("float64")
    combined["high"] = combined["high"].astype("float64")
    combined["low"] = combined["low"].astype("float64")
    combined["close"] = combined["close"].astype("float64")

    # ⬇️ critical fix: force column order to exactly match BRONZE_SCHEMA
    # before conversion, so positional fallback can't misalign columns
    combined = combined[[f.name for f in BRONZE_SCHEMA.fields]]

    combined_spark = spark.createDataFrame(combined, schema=BRONZE_SCHEMA)

    (
        combined_spark.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze_price_ohlcv")
    )

    print("✅ Data saved to bronze_price_ohlcv")
else:
    print("⚠️ No data to save.")

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 13, Finished, Available, Finished, False)

✅ Data saved to bronze_price_ohlcv


In [12]:
# ------------------------------------------------------------
# 8. Log ingestion
# ------------------------------------------------------------

ingestion_log = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "start_date": final_start,
    "end_date": today,
    "symbols_attempted": len(all_symbols),
    "symbols_successful": len(all_symbols) - len(failed_symbols),
    "symbols_failed": len(failed_symbols),
    "rows_inserted": len(combined),
    "status": "SUCCESS" if len(combined) > 0 else "FAILED"
}])

ingestion_log_spark = spark.createDataFrame(ingestion_log)

(
    ingestion_log_spark.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze_ingestion_log")
)

print("✅ Ingestion log saved to bronze_ingestion_log")

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 14, Finished, Available, Finished, False)

✅ Ingestion log saved to bronze_ingestion_log


In [13]:
display(
    spark.sql("""
        SELECT *
        FROM dbo.bronze_universe_snapshot
        LIMIT 50
    """)
)

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e578cb83-42f8-40a9-ac4a-3ad270fb40ec)

In [14]:
display(
    spark.sql("""
        SELECT
            u.symbol,
            u.company_name,
            COUNT(p.trade_date) AS price_days
        FROM bronze_universe_snapshot u
        LEFT JOIN bronze_price_ohlcv p
            ON u.symbol = p.symbol
        WHERE u.instrument_type = 'EQUITY'
          AND u.eligible_for_stock_screen = true
        GROUP BY u.symbol, u.company_name
        ORDER BY price_days ASC
    """)
)

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c093a71b-d437-4f6c-ae5d-780ea98682a9)

In [15]:
display(
    spark.sql("""
        SELECT *
        FROM bronze_price_ohlcv
        WHERE symbol = 'NIFTY50'
        ORDER BY trade_date DESC
        LIMIT 10
    """)
)

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2a4613d-9bbe-494e-a14a-b04c800ff72f)

In [16]:
display(
    spark.sql("""
        SELECT
            symbol,
            MAX(trade_date) AS latest_date
        FROM bronze_price_ohlcv
        GROUP BY symbol
        ORDER BY latest_date ASC
    """)
)

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac2ba0d9-a784-4837-ac71-4d922b623320)

In [17]:
from delta.tables import DeltaTable

combined_spark = spark.createDataFrame(combined)

if spark.catalog.tableExists("bronze_price_ohlcv"):
    target = DeltaTable.forName(spark, "bronze_price_ohlcv")
    (
        target.alias("t")
        .merge(
            combined_spark.alias("s"),
            "t.symbol = s.symbol AND t.trade_date = s.trade_date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    combined_spark.write.format("delta").saveAsTable("bronze_price_ohlcv")

StatementMeta(, 627343fa-d817-4e48-989c-137a91ed353f, 19, Finished, Available, Finished, False)